# InSight | Multi-Task DeBERTa-v3 with Post-Hoc Calibration (Platt & Temperature Scaling)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/visheshaggarwal0/InSight/blob/main/notebooks/02_deberta_multitask_sentiment_severity.ipynb)

### Objective
Standard neural networks are notoriously overconfident in their posterior probabilities. For an operational triaging system like InSight, an uncalibrated 95% confidence score could mean a false alarm or a missed critical P0 outage.

### Architecture:
1. **Backbone**: `microsoft/deberta-v3-small` with disentangled attention.
2. **Multi-Task Heads**:
   - Head A: 3-Class Sentiment (`Negative`, `Neutral`, `Positive`)
   - Head B: 4-Class Incident Severity (`P0 Blocker`, `P1 Critical`, `P2 Major`, `P3 Minor`)
3. **Loss Function**: Multi-task joint loss $\mathcal{L}_{total} = \mathcal{L}_{sent} + \lambda \mathcal{L}_{sev}$.
4. **Post-Hoc Calibration**: Temperature Scaling $z / T$ to minimize Expected Calibration Error (ECE) below 3%.


In [ ]:
# 1. Environment & CUDA Check
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

!pip install -q transformers datasets evaluate scikit-learn matplotlib seaborn


## 2. Multi-Task DeBERTa Model Architecture


In [ ]:
import torch.nn as nn
from transformers import DebertaV2Model, DebertaV2PreTrainedModel, AutoTokenizer

class MultiTaskDeBERTa(DebertaV2PreTrainedModel):
    def __init__(self, config, num_sentiment=3, num_severity=4):
        super().__init__(config)
        self.deberta = DebertaV2Model(config)
        self.dropout = nn.Dropout(0.2)
        
        # Dual classification heads
        self.sentiment_head = nn.Linear(config.hidden_size, num_sentiment)
        self.severity_head = nn.Linear(config.hidden_size, num_severity)
        
        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        # Mean pooling across non-padding tokens
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        pooled = sum_embeddings / sum_mask
        
        pooled = self.dropout(pooled)
        sentiment_logits = self.sentiment_head(pooled)
        severity_logits = self.severity_head(pooled)
        
        return sentiment_logits, severity_logits

print("MultiTaskDeBERTa class initialized successfully.")


## 3. Post-Hoc Temperature Scaling for Probability Calibration
Temperature scaling softens the softmax logits by a learned scalar parameter $T > 0$:
$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$
Optimized using Negative Log-Likelihood (NLL) on a held-out validation calibration split.


In [ ]:
import torch.optim as optim

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

    def fit_calibration(self, val_logits, val_labels):
        """Optimize temperature T to minimize NLL on validation logits."""
        nll_criterion = nn.CrossEntropyLoss()
        optimizer = optim.LBFGS([self.temperature], lr=0.01, max_iter=50)

        def eval_step():
            optimizer.zero_grad()
            loss = nll_criterion(self.forward(val_logits), val_labels)
            loss.backward()
            return loss

        optimizer.step(eval_step)
        print(f"Optimal Calibration Temperature T: {self.temperature.item():.4f}")
        return self.temperature.item()

def compute_ece(probs, labels, n_bins=10):
    """Expected Calibration Error (ECE)"""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = predictions == labels

    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            avg_confidence = np.mean(confidences[in_bin])
            avg_accuracy = np.mean(accuracies[in_bin])
            ece += np.abs(avg_confidence - avg_accuracy) * prop_in_bin

    return ece

# Demonstration of ECE calculation
sim_logits = torch.randn(500, 3) * 3.5  # Artificially overconfident logits
sim_labels = torch.randint(0, 3, (500,))

scaler = TemperatureScaler()
scaler.fit_calibration(sim_logits, sim_labels)

uncal_probs = torch.softmax(sim_logits, dim=-1).detach().numpy()
cal_probs = torch.softmax(scaler(sim_logits), dim=-1).detach().numpy()
labels_np = sim_labels.numpy()

uncal_ece = compute_ece(uncal_probs, labels_np)
cal_ece = compute_ece(cal_probs, labels_np)

print(f"Uncalibrated ECE: {uncal_ece * 100:.2f}%")
print(f"Calibrated ECE:   {cal_ece * 100:.2f}% (Reduced by {((uncal_ece - cal_ece)/uncal_ece)*100:.1f}%)")
